# Lab 3, Notebook B — Kernel ridge regression

*Linear Algebra for Data Science, UCU 2026. Uses Session 4.2 (kernels, the representer theorem, kernel ridge) and Session 4.1 (Cholesky).*

A kernel $k(\mathbf{x},\mathbf{y}) = \langle\phi(\mathbf{x}),\phi(\mathbf{y})\rangle$ is an inner product in a feature space $\mathcal{F}$ that you never construct. By the representer theorem, ridge regression in $\mathcal{F}$ has a solution $f(\mathbf{x}) = \sum_i\alpha_i k(\mathbf{x}_i,\mathbf{x})$ with

$$(K + \lambda I)\,\boldsymbol\alpha = \mathbf{y},\qquad K_{ij} = k(\mathbf{x}_i,\mathbf{x}_j).$$

$K+\lambda I$ is symmetric positive definite for every kernel and every $\lambda>0$, so one Cholesky factorization solves the problem. The cost depends on $n$, not on $\dim\mathcal{F}$. In this notebook you check the kernel trick, see what $\lambda$ does to the conditioning, and measure the cost.

**How to work.** As in Notebook A: **predict in writing, then compute, then explain any difference.**

**Rules.**
- Inside the functions you implement, do not call `np.linalg.inv` or `sklearn`'s `KernelRidge`. Use `KernelRidge` and `GaussianProcessRegressor` only where a cell says **check**.
- You may use `scipy.linalg.cholesky`, `cho_factor`/`cho_solve`, `solve_triangular` and `np.linalg.eigvalsh`.
- Do not change the seeds.
- Before you submit, run `Kernel → Restart & Run All`.

**Marking.** Core B is worth 2 points at the defence; the code point and the extension are shared with Notebook A.

**Contents.** B1 The solver · B2 The kernel trick, checked · B3 The Gaussian kernel and the role of $\lambda$ · B4 Cost and scale · Extension E3 (GP regression). Attempt **at most one** extension in the whole lab.

| | |
|---|---|
| **Team member 1** | *name, surname* |
| **Team member 2** | *name, surname* |
| **Extension attempted** (at most one in the whole lab) | *none / E1 / E2 / E3* |
| **Late days used on this lab** | *0* |
| **Runs top to bottom from a clean kernel** | *yes / no* |

In [ ]:
import sys, time
from math import comb
import numpy as np
import scipy, scipy.linalg as sl
import matplotlib, matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_friedman1
from sklearn.preprocessing import PolynomialFeatures

print("python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__, "| sklearn", sklearn.__version__)

**Data.** Friedman's benchmark #1: ten features uniform on $[0,1]$, of which only five matter, through
$y = 10\sin(\pi x_1x_2) + 20(x_3-\tfrac12)^2 + 10x_4 + 5x_5 + \text{noise}$, with noise standard deviation 1. It is synthetic, but nonlinear in a way a linear model cannot capture, which is the point of a kernel.

We split 1000 points into 500 for training, 250 for validation and 250 for testing. We standardise the features with **training** statistics only, and centre $y$ by its training mean: kernel ridge has no intercept, so predictions are of the centred $y$ (add the mean back to report them in the original units).

In [ ]:
X_all, y_all = make_friedman1(n_samples=1000, noise=1.0, random_state=0)
mu_x, sd_x = X_all[:500].mean(axis=0), X_all[:500].std(axis=0)
X_all = (X_all - mu_x) / sd_x
y_mean = y_all[:500].mean()
Xtr, Xva, Xte = X_all[:500], X_all[500:750], X_all[750:]
ytr, yva, yte = y_all[:500] - y_mean, y_all[500:750] - y_mean, y_all[750:] - y_mean
print(Xtr.shape, Xva.shape, Xte.shape)

def mse(a, b):
    return np.mean((a - b) ** 2)

---
## B1. The solver

Implement:
- `k_poly(X, Y, d, c)`, the matrix of $(c + \mathbf{x}^\top\mathbf{y})^d$ over the rows of `X` and `Y`;
- `k_gauss(X, Y, sigma)`, the matrix of $\exp(-\|\mathbf{x}-\mathbf{y}\|^2/(2\sigma^2))$. Use $\|\mathbf{x}-\mathbf{y}\|^2 = \|\mathbf{x}\|^2+\|\mathbf{y}\|^2-2\mathbf{x}^\top\mathbf{y}$, with no Python loops over points, and clip tiny negative values caused by rounding to zero;
- `krr_fit(K, y, lam)`, which returns $\boldsymbol\alpha$ solving $(K+\lambda I)\boldsymbol\alpha = \mathbf{y}$ by one Cholesky factorization, leaving `K` unchanged;
- `krr_predict(K_test_train, alpha)`.

In [ ]:
def k_poly(X, Y, d, c):
    # YOUR CODE HERE
    raise NotImplementedError

def k_gauss(X, Y, sigma):
    # YOUR CODE HERE
    raise NotImplementedError

def krr_fit(K, y, lam):
    # YOUR CODE HERE
    raise NotImplementedError

def krr_predict(K_test_train, alpha):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check against scikit-learn on 200 training points (sklearn calls lam 'alpha', and uses gamma = 1/(2 sigma^2))
from sklearn.kernel_ridge import KernelRidge
sigma, lam = 4.0, 0.1
a = krr_fit(k_gauss(Xtr[:200], Xtr[:200], sigma), ytr[:200], lam)
mine = krr_predict(k_gauss(Xva, Xtr[:200], sigma), a)
ref = KernelRidge(alpha=lam, kernel="rbf", gamma=1 / (2 * sigma ** 2)).fit(Xtr[:200], ytr[:200]).predict(Xva)
print("max difference from sklearn:", np.abs(mine - ref).max())
assert np.allclose(mine, ref)

---
## B2. The kernel trick, checked

**(a)** For the polynomial kernel $(1+\mathbf{x}^\top\mathbf{y})^2$ on $\mathbb{R}^{10}$, write the explicit feature map `phi_poly2(X)` with $\phi(\mathbf{x})^\top\phi(\mathbf{y}) = (1+\mathbf{x}^\top\mathbf{y})^2$. Expand the square to find the entries, minding the $\sqrt2$ factors. How many features are there? Check against `k_poly`.

In [ ]:
def phi_poly2(X):
    # YOUR CODE HERE
    raise NotImplementedError

Phi_tr, Phi_te = phi_poly2(Xtr), phi_poly2(Xte)
print("dim F =", Phi_tr.shape[1])
# check
assert np.allclose(Phi_tr[:50] @ Phi_te[:40].T, k_poly(Xtr[:50], Xte[:40], 2, 1))

**Your answer.** *Write it here.*

**(b)** Fit ordinary ridge regression on the features: $\mathbf{w}$ solving $(\Phi^\top\Phi+\lambda I)\mathbf{w} = \Phi^\top\mathbf{y}$, by Cholesky, with **no separate intercept** and $\lambda = 1$. Compare its test predictions $\Phi_{\text{test}}\mathbf{w}$ with kernel ridge using `k_poly(·,·,2,1)` and the same $\lambda$. Explain why they agree, citing the representer theorem.

In [ ]:
lam = 1.0
# YOUR CODE HERE
raise NotImplementedError
print("max |feature route - kernel route| =", np.abs(pred_feat - pred_kern).max())
print("test MSE, degree-2 polynomial model:", mse(pred_kern, yte))

**Your answer.** *Write it here.*

**(c)** *Predict, then compute.* Count the flops of each route as functions of $n$, $p = 10$ and $D = \dim\mathcal{F}$:
- **feature route**: form $\Phi^\top\Phi$, about $nD^2$, plus a Cholesky, about $D^3/3$;
- **kernel route**: form $K$, about $n^2p$, plus a Cholesky, about $n^3/3$.

For degree $d$ on $\mathbb{R}^{10}$, $D = \binom{10+d}{d}$. *Predict* the degree at which the feature route becomes more expensive at $n=500$. Then time both for $d = 2,3,4,5$. For timing only, use `PolynomialFeatures(d)` to build a feature matrix of the right size; its weights differ from the exact $\phi$, which does not matter for the cost.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
def best_time(f, repeat=3):
    best = np.inf
    for _ in range(repeat):
        t0 = time.perf_counter(); f(); best = min(best, time.perf_counter() - t0)
    return best

def feature_route(d):
    P = PolynomialFeatures(d).fit_transform(Xtr)
    G = P.T @ P + np.eye(P.shape[1])
    return sl.cho_solve(sl.cho_factor(G, lower=True), P.T @ ytr)

n = len(Xtr)
feature_route(2); krr_fit(k_poly(Xtr, Xtr, 2, 1.0), ytr, 1.0)      # warm-up: first calls carry one-off overheads
for d in [2, 3, 4, 5]:
    D = comb(10 + d, d)
    flops_feat, flops_kern = n * D ** 2 + D ** 3 / 3, n ** 2 * 10 + n ** 3 / 3
    # YOUR CODE HERE
    raise NotImplementedError
    print(f"d = {d}: D = {D:5d} | predicted flops feature {flops_feat:.1e}, kernel {flops_kern:.1e} | "
          f"time feature {1e3*t_feat:8.1f} ms, kernel {1e3*t_kern:6.1f} ms")

**Your answer.** *Write it here.*

---
## B3. The Gaussian kernel and the role of $\lambda$

First choose $\sigma$ by a short search on the validation set, with $\lambda = 10^{-2}$ fixed.

In [ ]:
sigmas = [1, 2, 3, 4, 6, 8]
val_err = []
for s in sigmas:
    a = krr_fit(k_gauss(Xtr, Xtr, s), ytr, 1e-2)
    val_err.append(mse(krr_predict(k_gauss(Xva, Xtr, s), a), yva))
sigma = sigmas[int(np.argmin(val_err))]
print("validation MSE:", dict(zip(sigmas, np.round(val_err, 3))), "-> sigma =", sigma)
K = k_gauss(Xtr, Xtr, sigma)

**(a)** Compute the eigenvalues of `K` with `eigvalsh`. *Predict, then check:* does `cholesky(K)` succeed on the full ten-feature data?

**(b)** Now repeat with a **single** feature, $x_4$, so that the same 500 points lie on a line, using the same $\sigma$. *Predict before running:* will the eigenvalues decay faster or slower than in (a), and will `cholesky(K1)` succeed?
- Plot both spectra on one log scale (absolute values), and run both checks.
- In theory both matrices are PSD. Explain what the smallest computed eigenvalues look like, why, and why the dimension of the data made the difference.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
K1 = k_gauss(Xtr[:, [3]], Xtr[:, [3]], sigma)
for name, M in [("10 features", K), ("1 feature ", K1)]:
    ev = np.linalg.eigvalsh(M)
    try:
        sl.cholesky(M, lower=True); ok = "succeeds"
    except np.linalg.LinAlgError:
        ok = "FAILS"
    print(f"{name}: eigenvalues in [{ev[0]: .2e}, {ev[-1]:.1f}], cholesky {ok}")

plt.figure(figsize=(6, 3.5))
plt.semilogy(np.abs(np.linalg.eigvalsh(K))[::-1], label="10 features")
plt.semilogy(np.abs(np.linalg.eigvalsh(K1))[::-1], label="feature $x_4$ only")
plt.xlabel("index k"); plt.ylabel(r"$|\lambda_k(K)|$"); plt.legend(); plt.tight_layout(); plt.show()

**Your answer.** *Write it here.*

Show that `cholesky(K1 + 1e-12 I)` succeeds. Then fit kernel ridge on the single feature with $\lambda = 10^{-12}$ and with $\lambda = 10^{-1}$. For each, report $\|\boldsymbol\alpha\|$, the validation MSE, and how much the validation predictions move when $\mathbf{y}$ is perturbed by noise of size $10^{-6}$ (seed below). Explain why the first success does **not** mean the regression problem is solved well.

In [ ]:
sl.cholesky(K1 + 1e-12 * np.eye(len(K1)), lower=True)
print("cholesky(K1 + 1e-12 I) succeeds")
K1va = k_gauss(Xva[:, [3]], Xtr[:, [3]], sigma)
dy = 1e-6 * np.random.default_rng(0).standard_normal(len(ytr))
for lam in [1e-12, 1e-1]:
    # YOUR CODE HERE
    raise NotImplementedError

**Your answer.** *Write it here.*

**(c)** *Predict, then compute.* For $\lambda = 10^{-8}, 10^{-7},\dots,10^{2}$, predict $\operatorname{cond}(K+\lambda I)$ for the ten-feature `K` from its eigenvalues alone. Then compute it with `np.linalg.cond` and compare.

**Your prediction** *(write it before running the next cell)*:

In [ ]:
ev = np.linalg.eigvalsh(K)
lams = 10.0 ** np.arange(-8, 3)
for lam in lams:
    # YOUR CODE HERE
    raise NotImplementedError
    print(f"lam = {lam:7.0e}: predicted {pred:.3e}, computed {np.linalg.cond(K + lam * np.eye(len(K))):.3e}")

**Your answer.** *Write it here.*

**(d)** For the same values of $\lambda$, plot training and validation MSE (log scale in $\lambda$) and mark the $\lambda$ you would choose. Report the test MSE at that $\lambda$. In one paragraph, relate the left end of the plot to (c), and the right end to what ridge does to the coefficients.

In [ ]:
Kva = k_gauss(Xva, Xtr, sigma)
tr_err, va_err = [], []
for lam in lams:
    # YOUR CODE HERE
    raise NotImplementedError
best = lams[int(np.argmin(va_err))]
a = krr_fit(K, ytr, best)
print(f"chosen lam = {best:g}; test MSE = {mse(krr_predict(k_gauss(Xte, Xtr, sigma), a), yte):.3f}  (noise variance is 1)")
plt.figure(figsize=(6, 3.5))
plt.loglog(lams, tr_err, "o-", label="training"); plt.loglog(lams, va_err, "s-", label="validation")
plt.axvline(best, color="grey", ls=":", label="chosen")
plt.xlabel(r"$\lambda$"); plt.ylabel("MSE"); plt.legend(); plt.tight_layout(); plt.show()

**Your answer.** *Write it here.*

---
## B4. Cost and scale

**(a)** *Predict, then measure.* Predict the slope of $\log(\text{time})$ against $\log n$ separately for **building** $K$ (with `k_gauss`) and for **fitting** (with `krr_fit`). Time both for $n = 500, 1000, 2000, 4000, 8000$, on fresh Friedman data with the same distribution, and fit the slopes from $n \ge 1000$. *The largest case needs about 2 GB of memory and a few seconds.*

**Your prediction** *(write it before running the next cell)*:

In [ ]:
Xbig, ybig = make_friedman1(n_samples=8000, noise=1.0, random_state=1)
Xbig = (Xbig - Xbig.mean(axis=0)) / Xbig.std(axis=0)
ns = [500, 1000, 2000, 4000, 8000]
t_build, t_fit = [], []
for m in ns:
    t0 = time.perf_counter(); Km = k_gauss(Xbig[:m], Xbig[:m], sigma); t_build.append(time.perf_counter() - t0)
    t0 = time.perf_counter(); krr_fit(Km, ybig[:m] - ybig[:m].mean(), 1e-2); t_fit.append(time.perf_counter() - t0)
    del Km
    print(f"n = {m:5d}: build {t_build[-1]:7.3f} s, fit {t_fit[-1]:7.3f} s")
sb = np.polyfit(np.log(ns[1:]), np.log(t_build[1:]), 1)[0]
sf = np.polyfit(np.log(ns[1:]), np.log(t_fit[1:]), 1)[0]
print(f"fitted slopes: build {sb:.2f}, fit {sf:.2f}")

**Your answer.** *Write it here.*

**(b)** At 8 bytes per entry, at what $n$ does $K$ alone fill 16 GB? Name one thing you would have to give up, or change, to fit kernel ridge on ten times that many points. Session 6 returns to this; a sentence is enough.

**Your answer.** *Write it here.*

---
# Extension — attempt **at most one** in the whole lab

End your extension with a paragraph headed **Finding**: one claim you can defend at the defence, supported by a number in this notebook.

## E3. Gaussian-process regression is kernel ridge

A Gaussian process with kernel $k$ treats the values $f(\mathbf{x})$ as jointly Gaussian with mean 0 and covariance $k(\mathbf{x},\mathbf{x}')$. Observations are $y_i = f(\mathbf{x}_i) + \text{noise}$, with independent noise of variance $s^2$.

**(a)** Write down the joint covariance of $(\mathbf{y}, f(\mathbf{x}_*))$ for a test point $\mathbf{x}_*$. Use the conditional-Gaussian formula of Session 4.2 (frame `gauss-cond`: $\mathbf{x}_2\mid\mathbf{x}_1=\mathbf{a}$ has mean $\boldsymbol\mu_2 + \Sigma_{21}\Sigma_{11}^{-1}(\mathbf{a}-\boldsymbol\mu_1)$ and covariance $\Sigma/\Sigma_{11}$) to show that the posterior mean of $f(\mathbf{x}_*)$ equals the kernel ridge prediction with $\lambda = s^2$.

**Your answer.** *Write it here.*

**(b)** Implement `gp_posterior(Xtr, ytr, Xte, sigma, s2)` for the Gaussian kernel, returning the posterior mean and variance at each test point. Use one Cholesky factor of $K+s^2I$ and **one triangular solve** for all the variances (no inverse). **Check** the mean against your `krr_fit`/`krr_predict`, and both outputs against `GaussianProcessRegressor` with the kernel fixed.

The data below are one-dimensional, with a deliberate gap in $[0.4, 0.6]$.

In [ ]:
rng = np.random.default_rng(7)
x1 = np.concatenate([rng.uniform(0, 0.4, 20), rng.uniform(0.6, 1.0, 20)])[:, None]
y1 = np.sin(2 * np.pi * x1[:, 0]) + 0.5 * x1[:, 0] + 0.2 * rng.standard_normal(40)
xg = np.linspace(-0.2, 1.2, 300)[:, None]
sig_gp, s2 = 0.1, 0.2 ** 2

def gp_posterior(Xtr, ytr, Xte, sigma, s2):
    # YOUR CODE HERE
    raise NotImplementedError

mean, var = gp_posterior(x1, y1, xg, sig_gp, s2)
# check
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
gpr = GaussianProcessRegressor(kernel=RBF(length_scale=sig_gp), alpha=s2, optimizer=None).fit(x1, y1)
m_ref, sd_ref = gpr.predict(xg, return_std=True)
assert np.allclose(mean, krr_predict(k_gauss(xg, x1, sig_gp), krr_fit(k_gauss(x1, x1, sig_gp), y1, s2)))
assert np.allclose(mean, m_ref) and np.allclose(np.sqrt(np.clip(var, 0, None)), sd_ref, atol=1e-6)
print("posterior mean = kernel ridge prediction; matches sklearn")

**(c)** Plot the posterior mean with a band of $\pm2$ posterior standard deviations, together with the data. Explain where the band is wide and why, using the Schur-complement reading of the conditional covariance.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Your answer.** *Write it here.*

---
**Before you submit:** run `Kernel → Restart & Run All`, check that every written answer is filled in, and make sure your extension (if any) ends with a **Finding**.